# Milestone 1: Email Triage System
This notebook implements a simple rule-based email triage system. 
The goal is to classify emails into three categories:
- **notify_human**: Emails that require immediate human attention (e.g., security issues, password resets)
- **respond_or_act**: Emails that require action but not urgent (e.g., invoice, meeting reminders)
- **ignore**: Emails that can be safely ignored (e.g., promotions, newsletters)


In [1]:
import pandas as pd
import nltk
import re

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\skgha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load the dataset
We load the email dataset provided in `data/sample_emails_with_triage_200.csv` 
and display the first few rows using `df.head()` to understand the structure.


In [8]:
#2. Load Dataset
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


# Clean the email text
We create a new column `clean_text` by:
1. Converting all text to lowercase
2. Removing all non-alphabetic characters
This helps standardize the text for the triage rules.


In [9]:
#3. Text Cleaning

df['clean_text'] = (
    df['body']
    .astype(str)
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

df[['body','clean_text']].head()


,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr is due on please pay to ...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


# Define triage rules
The `triage_rule` function classifies each email based on keywords:
- High priority keywords → notify_human
- Promotional keywords → ignore
- Other action-required keywords → respond_or_act

We apply this function to the `clean_text` column to get the `triage` label.


In [10]:
def triage_rule(text):
    t = str(text)

    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return 'notify_human'

    if any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return 'ignore'

    if any(k in t for k in ['invoice','payment','overdue','due on','meeting']):
        return 'respond_or_act'

    return 'respond_or_act'


In [11]:
def read_calender() ->str:
    return "No Meeting scehduled"

def send_auto_reply() -> str :
    return "auto reply sent"

In [12]:
# Add this at the top of your notebook, with other imports
from typing import Dict, List

# Rule-based agent to react to email
def react_agent(email: Dict) -> Dict:
    thought = "Classify email intent using triage rules."
    action = triage_rule(email["body"])  # Use triage_rule
    if action == "respond_or_act":
        observation = send_auto_reply()
    elif action == "notify_human":
        observation = "Human notified."
    else:
        observation = "Email ignored."
    return {
        "thought": thought,
        "action": action,
        "observation": observation
    }


In [13]:
# 5. Run Agent on Dataset
results: List[Dict] = []

for _, row in df.iterrows():
    agent_result = react_agent(row)  # Fixed syntax
    results.append({
        "id": row["id"],
        "true_triage": row.get("true_triage", None),
        "predicted_triage": agent_result["action"]
    })

results_df = pd.DataFrame(results)
results_df.head()


,id,true_triage,predicted_triage
0,1,None,respond_or_act
1,2,None,respond_or_act
2,3,None,respond_or_act
3,4,None,respond_or_act
4,5,None,respond_or_act


# View triage results
We display the first few rows of `clean_text` and `triage` columns 
to verify that the rules are applied correctly.


In [14]:
df['triage'] = df['clean_text'].apply(triage_rule)
df[['clean_text','triage']].head()

#The triage function is applied to the cleaned text

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,respond_or_act
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


# Save the final output
We save the resulting DataFrame with the `triage` column to:
`data/milestone1_ayesha-naaz.csv` for submission.

In [15]:
df.to_csv("../data/milestone1_ayesha-naaz.csv", index=False)
print("Saved milestone1_ayesha-naaz.csv")

#the results are saved into: data/milestone1_ayesha-naaz.csv

Saved milestone1_ayesha-naaz.csv
